In [1]:
from __future__ import annotations

from datetime import date
from pathlib import Path
import json
import pandas as pd
import numpy as np


### Parte 1 - Diagnóstico

In [2]:
print("pandas:", pd.__version__)
print("cwd idea:", Path.cwd())
DATA_DIR = Path("Datos")
print("Datos existe:", DATA_DIR.exists())

pandas: 3.0.5
cwd idea: c:\Users\Guille\Documents\Máster\Desarrollo de Soluciones de IA\GitHub\dsia-26-27-GZC\1_programacion_avanzada_python
Datos existe: True


### Parte 2 Validación

In [3]:
def cargar_ventas(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"No existe el fichero: {path}")
    return pd.read_csv(path)


raw = cargar_ventas(DATA_DIR / "ventas.csv")
raw

,fecha,region,producto,unidades,precio_unitario,cliente_id
0,2026-01-09,Sur,Cable Kit,7,12.50,C005
1,2026-03-14,Oeste,Gateway,25,120.00,C032
2,2026-01-26,Oeste,Hub IoT,1,88.96,C016
3,2026-02-05,Norte,Gateway,0,120.00,C010
4,2026-02-07,Norte,Hub IoT,15,85.00,C027
...,...,...,...,...,...,...
145,2026-02-15,Sur,Sensor A,16,19.50,C027
146,2026-01-13,Oeste,Hub IoT,19,85.00,C040
147,2026-02-09,Oeste,Sensor B,9,32.00,C009
148,2026-01-08,Oeste,Sensor B,5,32.31,C024


In [4]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   fecha            150 non-null    str    
 1   region           150 non-null    str    
 2   producto         150 non-null    str    
 3   unidades         148 non-null    str    
 4   precio_unitario  149 non-null    float64
 5   cliente_id       150 non-null    str    
dtypes: float64(1), str(5)
memory usage: 7.2 KB


In [5]:
raw.shape

(150, 6)

In [6]:
raw.dtypes

fecha                  str
region                 str
producto               str
unidades               str
precio_unitario    float64
cliente_id             str
dtype: object

In [7]:
raw.isna().sum()

fecha              0
region             0
producto           0
unidades           2
precio_unitario    1
cliente_id         0
dtype: int64

In [8]:
def validar_ventas(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    # Haciendo una copia del DataFrame para no modificar el original
    df = frame.copy()
    # Convertir las columnas "unidades" y "precio_unitario" a tipo numérico
    df["unidades"] = pd.to_numeric(df["unidades"], errors="coerce")
    df["precio_unitario"] = pd.to_numeric(df["precio_unitario"], errors="coerce")

    # Crear una variable con True y False para identificar si la fila es valida
    df['valido'] = (
        df["unidades"].notna()
        & (df["unidades"] > 0)
        & df["precio_unitario"].notna()
        & (df["precio_unitario"] > 0)
    )

    # Crear dos dataframes distintos para los valores validos y los que contienen errores
    # eliminando la nueva columna de validacon
    validos = df[df['valido']==True].drop(columns=['valido'])
    errores = df[df['valido']==False].drop(columns=['valido'])
    validos["importe"] = validos["unidades"] * validos["precio_unitario"]
    return validos, errores


validos, errores = validar_ventas(raw)
print(f"válidas: {len(validos)} | inválidas: {len(errores)}")
errores

válidas: 140 | inválidas: 10


,fecha,region,producto,unidades,precio_unitario,cliente_id
3,2026-02-05,Norte,Gateway,0.0,120.0,C010
41,2026-02-09,Oeste,Cable Kit,4.0,0.0,C015
54,2026-02-01,Sur,Sensor B,NaN,32.0,C004
63,2026-02-11,Norte,Sensor A,NaN,19.5,C018
66,2026-02-17,Oeste,Hub IoT,5.0,NaN,C025
84,2026-02-15,Este,Gateway,NaN,120.0,C022
88,2026-02-07,Este,Hub IoT,-2.0,85.0,C012
109,2026-02-19,Norte,Cable Kit,0.0,-1.0,C030
134,2026-02-03,Sur,Sensor A,3.0,-5.0,C007
149,2026-02-13,Sur,Sensor B,8.0,-32.0,C020


In [9]:
validos.info()

<class 'pandas.DataFrame'>
Index: 140 entries, 0 to 148
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   fecha            140 non-null    str    
 1   region           140 non-null    str    
 2   producto         140 non-null    str    
 3   unidades         140 non-null    float64
 4   precio_unitario  140 non-null    float64
 5   cliente_id       140 non-null    str    
 6   importe          140 non-null    float64
dtypes: float64(3), str(4)
memory usage: 8.8 KB


### Parte 3 - Agregaciones
Importe total por region (ordenado desc).

In [10]:
# Importe total por region (orden descendente)
validos.groupby("region")["importe"].sum().sort_values(ascending=False)

region
Este     33122.53
Norte    22914.63
Sur      21396.35
Oeste    19164.66
Name: importe, dtype: float64

Top 3 producto por importe.

In [11]:
# Comprobando cuantos productos distintos hay
validos['producto'].value_counts()

producto
Cable Kit    31
Hub IoT      28
Sensor A     28
Sensor B     27
Gateway      26
Name: count, dtype: int64

In [12]:
# Top 3 productos por importe total (orden descendente).
validos.groupby("producto")["importe"].sum().sort_values(ascending=False).head(3)

producto
Gateway     40440.00
Hub IoT     33058.66
Sensor B    11568.66
Name: importe, dtype: float64

Lista por escrito (comentario o markdown) qué filas parecen inválidas y por qué.

In [13]:
errores

,fecha,region,producto,unidades,precio_unitario,cliente_id
3,2026-02-05,Norte,Gateway,0.0,120.0,C010
41,2026-02-09,Oeste,Cable Kit,4.0,0.0,C015
54,2026-02-01,Sur,Sensor B,NaN,32.0,C004
63,2026-02-11,Norte,Sensor A,NaN,19.5,C018
66,2026-02-17,Oeste,Hub IoT,5.0,NaN,C025
84,2026-02-15,Este,Gateway,NaN,120.0,C022
88,2026-02-07,Este,Hub IoT,-2.0,85.0,C012
109,2026-02-19,Norte,Cable Kit,0.0,-1.0,C030
134,2026-02-03,Sur,Sensor A,3.0,-5.0,C007
149,2026-02-13,Sur,Sensor B,8.0,-32.0,C020


### Parte 4 - Exportación

Escribe:
- Datos/ventas_limpias.csv
- Datos/calidad_datos.json con: 
    - _filas_totales, filas_validas, filas_invalidas, importe_total_

In [14]:
# Crear el PATH para los ficheros de salida
VALIDOS_PATH = DATA_DIR / "ventas_limpias.csv"
QUALITY_PATH = DATA_DIR / "calidad_datos.json"

In [15]:
# Exportar los datos limpios a un nuevo fichero CSV
validos.to_csv(VALIDOS_PATH, index=False)

In [16]:
# Crear una variable con la información
quality_report = {
    "filas_totales": len(raw),
    "filas_validas": len(validos),
    "filas_invalidas": len(errores),
    "importe_total": validos["importe"].sum(),
}

In [17]:
# Exportar los datos de calidad a un fichero JSON
with open(QUALITY_PATH, "w", encoding="utf-8") as f:
    json.dump(quality_report, f, indent=4)